# 01 - Data Preprocessing
This notebook orchestrates the end-to-end pipeline:
1. Loading raw CSVs per turbine.
2. Harmonizing sensors across Farms A, B, and C.
3. Feature engineering (trends, deltas, circular yaw).
4. Labeling failure windows.
5. Saving processed data to Parquet for efficient modeling.

## Setup Notebook

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
from pathlib import Path
import sys

from wtfd.data.preprocessing import WindFarmProcessor

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

## Setup Paths
Point to the raw data provided in the tree and set the destination for processed files.

In [3]:
project_root = Path(os.getcwd()).parent
RAW_DATA_ROOT = project_root / "data" / "raw" / "zenodo_windfarm_data"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
CONFIG_PATH = project_root / "config" / "feature_map.yaml"

# Ensure the processed directory exists
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Run Batch Processing
Initialize the processor and iterate through all 95 turbines. 
<br>This method is memory-safe as it processes and saves one turbine at a time.

In [4]:
processor = WindFarmProcessor(config_path=str(CONFIG_PATH), window_days=3)

print("Starting batch processing of 95 turbines...")
processor.process_all_turbines(
    raw_data_root=RAW_DATA_ROOT, 
    output_dir=PROCESSED_DATA_DIR
)
print("Batch processing complete. Files saved to data/processed/")

Starting batch processing of 95 turbines...

Processing Farm A:


Farm A: 100%|██████████████████████████████████████████████████████████████████████| 22/22 [00:13<00:00,  1.65turbine/s]



Processing Farm B:


Farm B: 100%|██████████████████████████████████████████████████████████████████████| 15/15 [00:17<00:00,  1.17s/turbine]



Processing Farm C:


Farm C: 100%|██████████████████████████████████████████████████████████████████████| 58/58 [03:22<00:00,  3.49s/turbine]

Batch processing complete. Files saved to data/processed/


## Load and Finalize the Dataset
Load the compressed Parquet files and perform imputation for features like `hydraulic_temp` that are missing in Farm B.

In [5]:
def get_final_dataset(processed_dir):
    files = list(Path(processed_dir).glob("*.parquet"))
    print(f"Loading {len(files)} processed files...")
    
    # Load and combine
    dfs = [pd.read_parquet(f) for f in files]
    full_df = pd.concat(dfs, ignore_index=True)
    
    # Final fleet-wide imputation (Site B hydraulic_temp)
    full_df = processor.impute_missing_sensors(full_df)
    
    # Sort for time-series consistency
    full_df = full_df.sort_values(['farm_id', 'asset_id', 'time_stamp'])
    return full_df

df_final = get_final_dataset(PROCESSED_DATA_DIR)

Loading 95 processed files...


## Data Quality Check
Verify the balance of our target class and ensure the harmonization worked.

In [6]:
print(f"Total Records: {len(df_final):,}")
print(f"Positive Samples (Failure Windows): {df_final['target'].sum():,}")
print(f"Class Imbalance Ratio: 1:{len(df_final)//df_final['target'].sum()}")

# Verify Farm B Imputation
site_b_check = df_final[df_final['farm_id'] == 'B']['has_hydraulic_temp'].unique()
print(f"Farm B hydraulic sensor indicator: {site_b_check} (Should be [0])")

df_final.head()

Total Records: 5,242,948
Positive Samples (Failure Windows): 4,761
Class Imbalance Ratio: 1:1101
Farm B hydraulic sensor indicator: [0] (Should be [0])


,time_stamp,asset_id,farm_id,amb_temp,has_amb_temp,wind_speed,has_wind_speed,pitch_angle,has_pitch_angle,active_power,...,vibration_raw,has_vibration_raw,hydraulic_temp,has_hydraulic_temp,temp_delta_gearbox,temp_delta_hydraulic,power_efficiency,temp_trend_24h,rpm_volatility,target
926040,2022-01-01 00:00:00,0,A,18.0,1,4.1,1,-0.3,1,37.662,...,13.8,1,32.0,1,27.0,14.0,0.546452,NaN,NaN,0
926041,2022-01-01 00:10:00,0,A,18.0,1,4.1,1,-0.3,1,31.730,...,7.1,1,32.0,1,27.0,14.0,0.460382,NaN,NaN,0
926042,2022-01-01 00:20:00,0,A,18.0,1,4.1,1,-0.3,1,34.488,...,9.5,1,32.0,1,27.0,14.0,0.500399,NaN,NaN,0
926043,2022-01-01 00:30:00,0,A,18.0,1,4.4,1,-0.5,1,44.568,...,25.4,1,32.0,1,27.0,14.0,0.523197,NaN,NaN,0
926044,2022-01-01 00:40:00,0,A,18.0,1,5.5,1,-1.5,1,99.174,...,32.8,1,32.0,1,28.0,14.0,0.596087,NaN,NaN,0


In [7]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5242948 entries, 926040 to 5075601
Data columns (total 33 columns):
 #   Column                Dtype         
---  ------                -----         
 0   time_stamp            datetime64[ns]
 1   asset_id              int64         
 2   farm_id               object        
 3   amb_temp              float64       
 4   has_amb_temp          int64         
 5   wind_speed            float64       
 6   has_wind_speed        int64         
 7   pitch_angle           float64       
 8   has_pitch_angle       int64         
 9   active_power          float64       
 10  has_active_power      int64         
 11  gen_speed             float64       
 12  has_gen_speed         int64         
 13  gearbox_oil_temp      float64       
 14  has_gearbox_oil_temp  int64         
 15  transformer_temp      float64       
 16  has_transformer_temp  int64         
 17  nacelle_temp          float64       
 18  has_nacelle_temp      int64         
 19  

In [8]:
df_final.describe()

,time_stamp,asset_id,amb_temp,has_amb_temp,wind_speed,has_wind_speed,pitch_angle,has_pitch_angle,active_power,has_active_power,...,vibration_raw,has_vibration_raw,hydraulic_temp,has_hydraulic_temp,temp_delta_gearbox,temp_delta_hydraulic,power_efficiency,temp_trend_24h,rpm_volatility,target
count,5242948,5.242948e+06,5.242948e+06,5242948.0,5.242948e+06,5242948.0,5.242948e+06,5242948.0,5.242948e+06,5242948.0,...,5.242948e+06,5242948.0,5.242948e+06,5.242948e+06,5.242660e+06,4.383595e+06,5.236118e+06,5.228980e+06,5.239623e+06,5.242948e+06
mean,2023-01-16 03:05:20.091330816,2.487676e+01,2.365264e+01,1.0,6.827278e+00,1.0,4.495359e+00,1.0,4.273028e+01,1.0,...,1.617368e+01,1.0,2.657636e+01,8.361485e-01,2.707901e+01,7.736005e-01,1.473930e-01,1.834301e-02,1.683464e+02,9.080769e-04
min,2022-01-01 00:00:00,0.000000e+00,-2.732000e+02,1.0,-1.490000e+01,1.0,-2.990000e+00,1.0,-1.073400e+01,1.0,...,-1.858250e-01,1.0,0.000000e+00,0.000000e+00,-2.817350e+03,-8.308000e+02,-7.246571e+03,-2.144269e+03,0.000000e+00,0.000000e+00
25%,2022-10-05 03:50:00,1.100000e+01,1.800000e+01,1.0,4.191000e+00,1.0,-1.800000e+00,1.0,3.290200e-02,1.0,...,1.742750e-01,1.0,2.009000e+01,1.000000e+00,1.880050e+01,-7.760000e+00,7.534264e-05,-3.137500e+00,4.578035e+01,0.000000e+00
50%,2023-01-15 22:20:00,2.100000e+01,2.500000e+01,1.0,6.680000e+00,1.0,-1.400000e+00,1.0,3.120400e-01,1.0,...,2.820250e-01,1.0,2.657636e+01,1.000000e+00,2.488900e+01,-4.300000e+00,6.287700e-04,-3.600000e-02,1.159875e+02,0.000000e+00
75%,2023-05-08 06:50:00,4.200000e+01,2.915400e+01,1.0,9.890000e+00,1.0,9.064000e-01,1.0,7.896585e-01,1.0,...,4.193333e+00,1.0,2.952900e+01,1.000000e+00,3.200000e+01,1.100000e+01,9.205175e-04,3.000000e+00,2.546583e+02,0.000000e+00
max,2024-02-06 16:30:00,5.600000e+01,1.044300e+03,1.0,6.000000e+01,1.0,9.387000e+01,1.0,7.425120e+02,1.0,...,8.232000e+02,1.0,1.511800e+03,1.000000e+00,4.312045e+02,1.488055e+03,3.813079e+05,2.136078e+03,8.942382e+02,1.000000e+00
std,NaN,1.778590e+01,7.932802e+00,0.0,5.920968e+00,0.0,1.598866e+01,0.0,1.336220e+02,0.0,...,5.991696e+01,0.0,8.975731e+00,3.701408e-01,2.337038e+01,1.215492e+01,1.716315e+02,9.690776e+00,1.635074e+02,3.012063e-02
